## Run recon-all

2025/05/08: Added FS8 recon-all

In [ ]:
import shutil
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 200)

In [ ]:
pipeline_version = "all"
freesurfer_module_version = "freesurfer/8.0.0"
pipeline_dirname = "recon-all-8"
ncpus = 10
zres_vals = [3, 5]

In [ ]:
project_dir = Path("/data/ABCD_MBDU/ohbm2024/")
bids_dir = Path("/data/ABCD_DSST/ABCD/imaging_data/fast_track/rawdata/")
fsres_dir = project_dir / "data/derivatives/freesurfer/"
swarm_cmd_dir = project_dir / "swarm/swarm_commands"
swarm_log_dir = project_dir / "swarm/swarm_logs"

balanced_scans = pd.read_csv(project_dir / "code/balanced_scans.csv")

In [ ]:
balanced_scans["path"] = bids_dir / balanced_scans.filename

In [ ]:
assert balanced_scans.path.apply(lambda x: x.exists()).all()

In [ ]:
t1w_scans = balanced_scans.query("modality == 'T1w'").loc[
    :, ["participant_id", "session_id", "path"]
]
t1w_scans = t1w_scans.rename(columns={"path": "t1_path"})

t2w_scans = balanced_scans.query("modality == 'T2w'").loc[
    :, ["participant_id", "session_id", "path"]
]
t2w_scans = t2w_scans.rename(columns={"path": "t2_path"})

scans_to_run = t1w_scans.merge(
    t2w_scans, how="left", on=["participant_id", "session_id"]
)
scans_to_run = (
    scans_to_run.rename(columns={"participant_id": "subject", "session_id": "session"})
    .groupby("session")
    .head(250)
)

In [ ]:
scans_to_run.head()

# Run freesurfer 8 recon-all on full res T1 + T2


In [ ]:
ncpus = 10
cmds = []

for _, row in scans_to_run.iterrows():
    fs_outdir = fsres_dir / pipeline_dirname / row.subject / row.session
    fs_outdir.mkdir(exist_ok=True, parents=True)

    cmd = f"""\
export SUBJECTS_DIR=/lscratch/$SLURM_JOBID/fs_out; \
mkdir $SUBJECTS_DIR; \
source $FREESURFER_HOME/SetUpFreeSurfer.sh; \
recon-all \
    -subjid {row.subject} \
    -openmp {ncpus} \
    -all \
    -i {row.t1_path} \
    -T2 {row.t2_path}; \
rsync -a /lscratch/$SLURM_JOBID/fs_out {fs_outdir}; \
chown -R :ABCD_MBDU {fs_outdir}; \
chmod -R g+rw {fs_outdir}; \
rm -rf /lscratch/$SLURM_JOBID/* \
"""

    cmds.append(cmd)

In [ ]:
cmds[:1]

In [ ]:
def get_fs_swarm_cmd(test=True):
    run_name = f"abcd_recon_{pipeline_version}-fs-8-leej3{'-test' if test else ''}"

    swarm_cmd_file = swarm_cmd_dir / run_name
    swarm_cmd_file.write_text("\n".join(cmds[:2] if test else cmds))
    swarm_exec = f"""swarm -f {swarm_cmd_file.resolve()} -g 100 -t {ncpus} --gres=lscratch:400 --module {freesurfer_module_version},fsl --time 12:00:00 --logdir {swarm_log_dir.resolve()} --job-name {run_name} --partition norm"""
    return swarm_exec


swarm_exec = get_fs_swarm_cmd()
swarm_exec

In [ ]:
job_id = !{swarm_exec}
print(job_id)

In [ ]:
!scancel 56353383

In [ ]:
full_swarm_exec = get_fs_swarm_cmd(False)
full_swarm_exec

In [ ]:
job_id = !{full_swarm_exec}
print(job_id)

# Rerun for many failed runs

In [ ]:
# extracted manually with output from processing notebook.
failed_subs = [
    "sub-NDARINV01NAYMZH",
    "sub-NDARINV04EUBGTM",
    "sub-NDARINV086U18RD",
    "sub-NDARINV08R2PTT1",
    "sub-NDARINV0B7UGM1D",
    "sub-NDARINV0JWTGKAD",
    "sub-NDARINV0R5220TJ",
    "sub-NDARINV0U317B9P",
    "sub-NDARINV0X45NBYM",
    "sub-NDARINV0Y8YJ2UR",
    "sub-NDARINV0YVKYMJX",
    "sub-NDARINV14C1N3KZ",
    "sub-NDARINV1AKMLL9A",
    "sub-NDARINV1EGW0J5N",
    "sub-NDARINV1F5JEP46",
    "sub-NDARINV1JXDFV9Z",
    "sub-NDARINV1KXK7MDF",
    "sub-NDARINV20GGC8X5",
    "sub-NDARINV21EBHGF9",
    "sub-NDARINV249JM0NY",
    "sub-NDARINV29P0F670",
    "sub-NDARINV2F51HZAP",
    "sub-NDARINV2FV9YY14",
    "sub-NDARINV2P5R504F",
    "sub-NDARINV2RD4CZ7T",
    "sub-NDARINV2T1G705T",
    "sub-NDARINV2WYPW02N",
    "sub-NDARINV2Z2HJFG1",
    "sub-NDARINV32KLKRZC",
    "sub-NDARINV330HNJEV",
    "sub-NDARINV3AGKBZAW",
    "sub-NDARINV3CW8YR0W",
    "sub-NDARINV3HEHP1P4",
    "sub-NDARINV45BG08PF",
    "sub-NDARINV47UWAATW",
    "sub-NDARINV47W6DHJC",
    "sub-NDARINV486UUF4D",
    "sub-NDARINV4PYRUPX6",
    "sub-NDARINV512RTCH1",
    "sub-NDARINV51ZUKMA3",
    "sub-NDARINV52AE3MBX",
    "sub-NDARINV52XG9LJ3",
    "sub-NDARINV592319RL",
    "sub-NDARINV5C9WJ4B3",
    "sub-NDARINV5G3DP835",
    "sub-NDARINV5N40CFL4",
    "sub-NDARINV5V2AYDE7",
    "sub-NDARINV60TVEFTU",
    "sub-NDARINV665J58DF",
    "sub-NDARINV68WXH24G",
    "sub-NDARINV69FLDML4",
    "sub-NDARINV6B3HFDAY",
    "sub-NDARINV6KEGPJ6J",
    "sub-NDARINV6X9U8P56",
    "sub-NDARINV71136NP6",
    "sub-NDARINV7FG8NTPP",
    "sub-NDARINV7G30XLFU",
    "sub-NDARINV7J9K5U1L",
    "sub-NDARINV7WT9690H",
    "sub-NDARINV8451EHXW",
    "sub-NDARINV85UUUHN0",
    "sub-NDARINV87L30DBB",
    "sub-NDARINV87WRYJ7F",
    "sub-NDARINV88V8C4GJ",
    "sub-NDARINV8CBT9W65",
    "sub-NDARINV8J5VU553",
    "sub-NDARINV8L7MBY64",
    "sub-NDARINV8MGGJ4FD",
    "sub-NDARINV8T2NUL38",
    "sub-NDARINV956C4ZGG",
    "sub-NDARINV95YMR61C",
    "sub-NDARINV974A9111",
    "sub-NDARINV9EYHTYJT",
    "sub-NDARINV9HB34LLU",
    "sub-NDARINV9L4P2RJJ",
    "sub-NDARINV9R0Z0T45",
    "sub-NDARINV9UV6PK7Y",
    "sub-NDARINVA3VX7WRD",
    "sub-NDARINVA4KJXLYH",
    "sub-NDARINVA4MKK2EJ",
    "sub-NDARINVA68WCRUL",
    "sub-NDARINVBBH4GW2D",
    "sub-NDARINVBG7FNXZB",
]

In [ ]:
ncpus = 10
rerun_cmds = []

for _, row in scans_to_run.iterrows():
    if row.subject not in failed_subs:
        continue
    fs_outdir = fsres_dir / pipeline_dirname / row.subject / row.session
    shutil.rmtree(fs_outdir.parent, ignore_errors=True)
    fs_outdir.mkdir(exist_ok=True, parents=True)

    cmd = f"""\
export SUBJECTS_DIR=/lscratch/$SLURM_JOBID/fs_out; \
mkdir $SUBJECTS_DIR; \
source $FREESURFER_HOME/SetUpFreeSurfer.sh; \
recon-all \
    -subjid {row.subject} \
    -openmp {ncpus} \
    -all \
    -i {row.t1_path} \
    -T2 {row.t2_path}; \
rsync -a /lscratch/$SLURM_JOBID/fs_out {fs_outdir}; \
chown -R :ABCD_MBDU {fs_outdir}; \
chmod -R g+rw {fs_outdir}; \
rm -rf /lscratch/$SLURM_JOBID/* \
"""

    rerun_cmds.append(cmd)

In [ ]:
rerun_cmds[:1]

In [ ]:
def get_fs_swarm_cmd(cmds, test=True):
    run_name = (
        f"abcd_recon_{pipeline_version}-fs-8-rerun-leej3{'-test' if test else ''}"
    )

    swarm_cmd_file = swarm_cmd_dir / run_name
    swarm_cmd_file.write_text("\n".join(cmds[:2] if test else cmds))
    swarm_exec = f"""swarm -f {swarm_cmd_file.resolve()} -g 50 -t {ncpus} --gres=lscratch:400 --module {freesurfer_module_version},fsl --time 60:00:00 --logdir {swarm_log_dir.resolve()} --job-name {run_name} --partition norm"""
    return swarm_exec


swarm_exec = get_fs_swarm_cmd(rerun_cmds)
swarm_exec

In [ ]:
job_id = !{swarm_exec}
print(job_id)

In [ ]:
!scancel 56475212

In [ ]:
full_swarm_exec = get_fs_swarm_cmd(rerun_cmds, False)
full_swarm_exec

In [ ]:
job_id = !{full_swarm_exec}
print(job_id)

## Results


In [ ]:
!tree {fsres_dir}/{pipeline_dirname}/sub-NDARINV00HEV6HB